In [7]:
# 03 - Build clean data

# Purpose: apply the decisions in `docs/decision-log.md` to a copy of the raw data. `raw_df` is never modified; every change is made to `clean_df`, and row counts are reconciled after each step.

In [8]:
import pandas as pd

DATA_PATH = "../data/bank-additional-full.csv"
raw_df = pd.read_csv(DATA_PATH, sep=";", dtype=str, keep_default_na=False)
raw_df.shape

(41188, 21)

In [9]:
# Clean step 1: remove exact duplicate rows (decision 3)
clean_df = raw_df.drop_duplicates().copy()

print(f"Rows before: {len(raw_df)}")
print(f"Rows after:  {len(clean_df)}")
print(f"Removed:     {len(raw_df) - len(clean_df)}")
print()

print("y counts, raw vs clean:")
print(pd.DataFrame({
    "raw": raw_df["y"].value_counts(),
    "clean": clean_df["y"].value_counts(),
}))
print()

print(f"raw_df still has {len(raw_df)} rows (unchanged)")

Rows before: 41188
Rows after:  41176
Removed:     12

y counts, raw vs clean:
       raw  clean
y                
no   36548  36537
yes   4640   4639

raw_df still has 41188 rows (unchanged)


In [10]:
# Clean step 2: convert numeric columns from text to numbers
numeric_columns = [
    "age", "duration", "campaign", "pdays", "previous",
    "emp.var.rate", "cons.price.idx", "cons.conf.idx", "euribor3m", "nr.employed",
]

clean_df[numeric_columns] = clean_df[numeric_columns].apply(pd.to_numeric, errors="raise")

print(f"Rows: {len(clean_df)}")
print()
print("Column types:")
print(clean_df.dtypes)
print()
print("Numeric columns, min / median / max:")
clean_df[numeric_columns].describe().T[["min", "50%", "max"]]

Rows: 41176

Column types:
age                 int64
job                   str
marital               str
education             str
default               str
housing               str
loan                  str
contact               str
month                 str
day_of_week           str
duration            int64
campaign            int64
pdays               int64
previous            int64
poutcome              str
emp.var.rate      float64
cons.price.idx    float64
cons.conf.idx     float64
euribor3m         float64
nr.employed       float64
y                     str
dtype: object

Numeric columns, min / median / max:


,min,50%,max
age,17.000,38.000,98.000
duration,0.000,180.000,4918.000
campaign,1.000,2.000,56.000
pdays,0.000,999.000,999.000
previous,0.000,0.000,7.000
emp.var.rate,-3.400,1.100,1.400
cons.price.idx,92.201,93.749,94.767
cons.conf.idx,-50.800,-41.800,-26.900
euribor3m,0.634,4.857,5.045
nr.employed,4963.600,5191.000,5228.100


In [11]:
# Clean step 3: pdays placeholder and previously_contacted (decision 2)
pdays_was_999 = clean_df["pdays"] == 999

clean_df["pdays"] = clean_df["pdays"].where(~pdays_was_999).astype("Int64")
clean_df["previously_contacted"] = clean_df["previous"] > 0

print(f"Rows: {len(clean_df)}")
print(f"pdays type: {clean_df['pdays'].dtype}")
print()

# Reconcile against the raw data: the only difference should be the 12 duplicates removed
raw_999 = (raw_df["pdays"] == "999").sum()
clean_999 = clean_df["pdays"].isna().sum()
raw_real = len(raw_df) - raw_999
clean_real = clean_df["pdays"].notna().sum()
print(f"pdays = 999 / now missing:  raw {raw_999}, clean {clean_999}")
print(f"pdays with a real value:    raw {raw_real}, clean {clean_real}")
print(f"Total difference: {(raw_999 - clean_999) + (raw_real - clean_real)} (should be 12)")
print()

# Rules from profiling should still hold
print("pdays missing vs previously_contacted:")
print(pd.crosstab(clean_df["pdays"].isna(), clean_df["previously_contacted"],
                  rownames=["pdays is missing"], colnames=["previously_contacted"]))
print()
print("poutcome = success rows with missing pdays (should be 0):",
      ((clean_df["poutcome"] == "success") & clean_df["pdays"].isna()).sum())
print()
print("pdays real values: min / median / max")
print(clean_df["pdays"].agg(["min", "median", "max"]))

Rows: 41176
pdays type: Int64

pdays = 999 / now missing:  raw 39673, clean 39661
pdays with a real value:    raw 1515, clean 1515
Total difference: 12 (should be 12)

pdays missing vs previously_contacted:
previously_contacted  False  True 
pdays is missing                  
False                     0   1515
True                  35551   4110

poutcome = success rows with missing pdays (should be 0): 0

pdays real values: min / median / max
min        0.0
median     6.0
max       27.0
Name: pdays, dtype: float64


In [12]:
# Clean step 4: rename columns to clear, readable names
column_names = {
    "age": "age",
    "job": "job",
    "marital": "marital_status",
    "education": "education",
    "default": "credit_default",
    "housing": "housing_loan",
    "loan": "personal_loan",
    "contact": "contact_type",
    "month": "contact_month",
    "day_of_week": "contact_weekday",
    "duration": "call_duration_seconds",
    "campaign": "contacts_this_campaign",
    "pdays": "days_since_previous_contact",
    "previous": "contacts_before_campaign",
    "poutcome": "previous_campaign_outcome",
    "emp.var.rate": "employment_variation_rate",
    "cons.price.idx": "consumer_price_index",
    "cons.conf.idx": "consumer_confidence_index",
    "euribor3m": "euribor_3_month_rate",
    "nr.employed": "number_employed",
    "y": "subscribed_term_deposit",
}

before_rename_df = clean_df.copy()
clean_df = clean_df.rename(columns=column_names)

# Prove only the labels changed: rename back and compare with the copy
reversed_names = {new: old for old, new in column_names.items()}
round_trip_matches = clean_df.rename(columns=reversed_names).equals(before_rename_df)

print(f"Rows: {len(clean_df)}, columns: {clean_df.shape[1]}")
print(f"Values identical after renaming back: {round_trip_matches}")
print()
print("New columns:")
print(list(clean_df.columns))

Rows: 41176, columns: 22
Values identical after renaming back: True

New columns:
['age', 'job', 'marital_status', 'education', 'credit_default', 'housing_loan', 'personal_loan', 'contact_type', 'contact_month', 'contact_weekday', 'call_duration_seconds', 'contacts_this_campaign', 'days_since_previous_contact', 'contacts_before_campaign', 'previous_campaign_outcome', 'employment_variation_rate', 'consumer_price_index', 'consumer_confidence_index', 'euribor_3_month_rate', 'number_employed', 'subscribed_term_deposit', 'previously_contacted']


In [13]:
# Clean step 5: save the clean data and prove it saved correctly
import os

CLEAN_PATH = "../data/bank_marketing_clean.parquet"

clean_df = clean_df.reset_index(drop=True)
clean_df.to_parquet(CLEAN_PATH, index=False)

reloaded_df = pd.read_parquet(CLEAN_PATH)

print(f"Saved to: {CLEAN_PATH}")
print(f"File size: {os.path.getsize(CLEAN_PATH) / 1024:.0f} KB")
print(f"Shape saved: {clean_df.shape}, shape reloaded: {reloaded_df.shape}")
print(f"Values identical after reload: {clean_df.equals(reloaded_df)}")
print(f"Column types identical after reload: {(clean_df.dtypes == reloaded_df.dtypes).all()}")

Saved to: ../data/bank_marketing_clean.parquet
File size: 232 KB
Shape saved: (41176, 22), shape reloaded: (41176, 22)
Values identical after reload: True
Column types identical after reload: True
